# Combined Grain Product Strategy

This notebook combines the final corn, soybean, and wheat signals into one tradeable product book.

Rules:
- no fitted portfolio optimizer;
- every promoted sleeve uses the Cargill dataset;
- product sleeves are equal-weighted;
- compare two implementations: outright futures only, and front-vs-second calendar spreads.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grain_futures_strategy import (
    backtest_positions_with_costs,
    build_calendar_spread_pnl,
    build_feature_panels,
    load_train_set,
    split_performance,
)
from grain_research import (
    backtest_positions_product_flow,
    build_corn_product_flow_signal_universe,
    build_product_flow_feature_panels,
    clean_product_flow_signal,
    corn_abundant_supply_masks,
    corn_positions_from_signal,
    mean_product_flow_signals,
    product_flow_performance_metrics,
)
from research_config import DEFAULT_MARGIN_PER_LOT, SPLIT_DATE
from wheat_improvement_experiment import (
    build_pair_price_trend_from_prices,
    build_pair_signals,
    positions_from_pair_signal,
)

DATA_DIR = "train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
DD_CAPITAL_USD = 10_000.0
TRADE_COST_PER_LOT = 8.75
HOLDING_COST_RATE = 0.05
PRODUCT_WEIGHT = 1.0 / 3.0

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Load Shared Data

In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl = build_feature_panels(data)
calendar_pnl = build_calendar_spread_pnl(data)
index = futures_pnl.index

cargill_columns = ["cgl_inventory_change", "crush_surprise", "crush_utilization"]
display(pd.DataFrame([
    {
        "commodity": commodity,
        "has_cargill_inputs": all(col in feature_panels[commodity].columns for col in cargill_columns),
        "start": index.min(),
        "end": index.max(),
        "oos_start": OOS_START,
    }
    for commodity in ["CORN", "SOYABEAN", "WHEAT_SRW", "WHEAT_HRW"]
]))

,commodity,has_cargill_inputs,start,end,oos_start
0,CORN,True,2010-01-04,2020-12-31,2018-01-01
1,SOYABEAN,True,2010-01-04,2020-12-31,2018-01-01
2,WHEAT_SRW,True,2010-01-04,2020-12-31,2018-01-01
3,WHEAT_HRW,True,2010-01-04,2020-12-31,2018-01-01


## 2. Common Helpers

In [3]:
def clean_signal(series, target_index=index):
    return (
        pd.Series(series, index=target_index)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(-5.0, 5.0)
    )


def positions_from_single_signal(signal, pnl, commodity, target_daily_vol=75.0, max_abs_lot=0.50):
    signal = clean_signal(signal, pnl.index)
    signal = pd.Series(np.tanh(signal / 2.0), index=pnl.index)
    signal = signal.ewm(halflife=3.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    asset_vol = pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    lots = (signal * float(target_daily_vol) / asset_vol).clip(-float(max_abs_lot), float(max_abs_lot)).fillna(0.0)
    return pd.DataFrame({commodity: lots}, index=pnl.index)


def active_metrics(bt, mask=None):
    sample_bt = bt if mask is None else bt.loc[mask]
    active = sample_bt["held_gross_exposure"] > 1.0e-12
    pnl = sample_bt.loc[active, "net_pnl"].dropna()
    if len(pnl) == 0:
        return {"days": 0, "total_pnl": 0.0, "sharpe": np.nan, "max_drawdown": 0.0, "hit_rate": np.nan}
    cum = pnl.cumsum()
    dd = cum - cum.cummax()
    vol = pnl.std()
    return {
        "days": int(len(pnl)),
        "total_pnl": float(pnl.sum()),
        "sharpe": np.nan if vol == 0.0 else float(pnl.mean() / vol * np.sqrt(252.0)),
        "max_drawdown": float(dd.min()),
        "hit_rate": float((pnl > 0.0).mean()),
        "turnover": float(sample_bt.loc[pnl.index, "turnover"].mean()),
        "avg_gross_exposure": float(sample_bt.loc[pnl.index, "gross_exposure"].mean()),
    }


def backtest_book(positions, pnl):
    return backtest_positions_with_costs(
        positions.reindex(pnl.index).fillna(0.0),
        pnl,
        trade_cost_per_lot=TRADE_COST_PER_LOT,
        holding_cost_rate=HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def result_row(test, bt):
    oos = active_metrics(bt, bt.index >= OOS_START)
    full = active_metrics(bt)
    return {
        "test": test,
        "oos_sharpe": oos["sharpe"],
        "oos_pnl": oos["total_pnl"],
        "oos_dd_pct": oos["max_drawdown"] / DD_CAPITAL_USD * 100.0,
        "oos_active_days": oos["days"],
        "full_sharpe": full["sharpe"],
        "full_pnl": full["total_pnl"],
        "full_dd_pct": full["max_drawdown"] / DD_CAPITAL_USD * 100.0,
        "hit_rate": oos["hit_rate"],
        "turnover": full["turnover"],
        "avg_gross_exposure": full["avg_gross_exposure"],
    }


def combine_equal_weight(sleeves, columns):
    out = pd.DataFrame(0.0, index=index, columns=columns)
    for sleeve in sleeves:
        aligned = sleeve.reindex(index).fillna(0.0)
        for column in aligned.columns:
            out[column] = out[column] + PRODUCT_WEIGHT * aligned[column]
    return out.fillna(0.0)

## 3. Final Corn Signal

In [4]:
corn_data = data
corn_feature_panels, corn_pnl_all = build_product_flow_feature_panels(corn_data)
corn_pnl = corn_pnl_all[["CORN"]]
corn_index = corn_pnl.index
corn_panel = corn_feature_panels["CORN"].reindex(corn_index).fillna(0.0)

corn_signals = {
    "mom_20": clean_product_flow_signal(corn_panel["mom_20"], corn_index),
    "mom_60": clean_product_flow_signal(corn_panel["mom_60"], corn_index),
    "rev_5": clean_product_flow_signal(corn_panel["rev_5"], corn_index),
    "mom_60_rev_5_equal": clean_product_flow_signal((corn_panel["mom_60"] + corn_panel["rev_5"]) / 2.0, corn_index),
}
trend_strength = corn_panel["mom_60"].abs()
trend_threshold = trend_strength.expanding(min_periods=252).median().shift(1)
trend_regime = (trend_strength > trend_threshold).fillna(False)
corn_signals["trend_mom_else_mr"] = clean_product_flow_signal(
    pd.Series(np.where(trend_regime, corn_panel["mom_60"], corn_panel["rev_5"]), index=corn_index),
    corn_index,
)


def corn_wf_signal(candidates, pnl, start=OOS_START):
    selected_signal = pd.Series(0.0, index=pnl.index)
    rows = []
    for rebalance in pd.date_range(pd.Timestamp(start), pnl.index.max(), freq="YS"):
        next_rebalance = rebalance + pd.DateOffset(years=1)
        train_end = rebalance - pd.DateOffset(years=2)
        train_mask = pnl.index < train_end
        validation_mask = (pnl.index >= train_end) & (pnl.index < rebalance)
        trade_mask = (pnl.index >= rebalance) & (pnl.index < next_rebalance)
        scored = []
        for name, signal in candidates.items():
            pos = corn_positions_from_signal(signal, pnl, mode="long_short")
            bt, _ = backtest_positions_product_flow(pos, pnl)
            train = product_flow_performance_metrics(bt.loc[train_mask])
            validation = product_flow_performance_metrics(bt.loc[validation_mask])
            train_sharpe = train.get("sharpe", np.nan)
            validation_sharpe = validation.get("sharpe", np.nan)
            eligible = pd.notnull(train_sharpe) and pd.notnull(validation_sharpe) and train_sharpe > 0.0 and validation_sharpe > 0.0
            scored.append({
                "rebalance": rebalance.date(),
                "candidate": name,
                "train_sharpe": train_sharpe,
                "validation_sharpe": validation_sharpe,
                "score": validation_sharpe + 0.25 * train_sharpe if eligible else -np.inf,
            })
        table = pd.DataFrame(scored)
        selected = table.sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
        selected_signal.loc[trade_mask] = candidates[selected].loc[trade_mask]
        rows.append({"year": rebalance.year, "selected": selected})
    return clean_product_flow_signal(selected_signal, pnl.index), pd.DataFrame(rows)


corn_wf, corn_wf_rows = corn_wf_signal(corn_signals, corn_pnl)
corn_signal_universe = build_corn_product_flow_signal_universe(corn_feature_panels, corn_pnl_all, DATA_DIR)
corn_cargill_physical = clean_product_flow_signal(
    mean_product_flow_signals([
        corn_signal_universe["given_cgl_inventory_pressure"],
        corn_signal_universe["given_cgl_crush_activity"],
    ], corn_index),
    corn_index,
)
corn_disagree = (corn_wf * corn_cargill_physical < 0.0) & (corn_wf.abs() > 0.05) & (corn_cargill_physical.abs() > 0.25)
corn_final_signal = corn_wf.where(~corn_disagree, 0.0)

corn_masks = corn_abundant_supply_masks(corn_data, corn_feature_panels, corn_pnl)
corn_outright_pos = corn_positions_from_signal(corn_final_signal, corn_pnl, mode="long_short")
corn_guard = corn_masks["below_ma_or_negative_mom"].reindex(corn_index).fillna(False)
corn_outright_pos.loc[corn_guard, "CORN"] = 0.0
corn_outright_pos = corn_outright_pos.reindex(index).fillna(0.0)

corn_calendar_signal = corn_final_signal.reindex(index).fillna(0.0)
corn_calendar_pos = positions_from_single_signal(corn_calendar_signal, calendar_pnl[["CORN"]], "CORN")
corn_calendar_pos.loc[corn_guard.reindex(index).fillna(False), "CORN"] = 0.0

display(corn_wf_rows)
display(pd.DataFrame({"final_rule": ["wf_momentum_mr + cargill_disagree_flat + below_ma_or_negative_mom_flat"]}))

,year,selected
0,2018,mom_60_rev_5_equal
1,2019,rev_5
2,2020,mom_20


,final_rule
0,wf_momentum_mr + cargill_disagree_flat + below...


## 4. Final Soybean Signal

In [5]:
soy_panel = feature_panels["SOYABEAN"].reindex(index).fillna(0.0)
soy_pnl = futures_pnl[["SOYABEAN"]]
soy_trend_gate = soy_panel["mom_60"].abs() > soy_panel["mom_60"].abs().expanding(min_periods=252).median().shift(1)

soy_candidates = {
    "mom20": clean_signal(soy_panel["mom_20"]),
    "mom60": clean_signal(soy_panel["mom_60"]),
    "mr5": clean_signal(soy_panel["rev_5"]),
    "mom60_mr5_equal": clean_signal((soy_panel["mom_60"] + soy_panel["rev_5"]) / 2.0),
    "trend_mom_else_mr": clean_signal(soy_panel["mom_60"].where(soy_trend_gate, soy_panel["rev_5"])),
}


def soy_wf_signal(candidates, pnl):
    selected_signal = pd.Series(0.0, index=pnl.index)
    rows = []
    backtests = {name: backtest_book(positions_from_single_signal(signal, pnl, "SOYABEAN"), pnl) for name, signal in candidates.items()}
    for year in range(OOS_START.year, pnl.index.max().year + 1):
        start = pd.Timestamp(f"{year}-01-01")
        end = pd.Timestamp(f"{year + 1}-01-01")
        validation_start = start - pd.DateOffset(years=2)
        train_mask = pnl.index < validation_start
        validation_mask = (pnl.index >= validation_start) & (pnl.index < start)
        trade_mask = (pnl.index >= start) & (pnl.index < end)
        scored = []
        for name, bt in backtests.items():
            train = active_metrics(bt, train_mask)
            validation = active_metrics(bt, validation_mask)
            scored.append({
                "year": year,
                "candidate": name,
                "train_sharpe": train["sharpe"],
                "validation_sharpe": validation["sharpe"],
                "score": validation["sharpe"] if train["sharpe"] > 0.0 else -np.inf,
            })
        table = pd.DataFrame(scored)
        selected = table.sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
        selected_signal.loc[trade_mask] = candidates[selected].loc[trade_mask]
        rows.append({"year": year, "selected": selected})
    return clean_signal(selected_signal, pnl.index), pd.DataFrame(rows)


soy_wf, soy_wf_rows = soy_wf_signal(soy_candidates, soy_pnl)
soy_cargill = clean_signal((-soy_panel["cgl_inventory_change"] + soy_panel["crush_surprise"] + soy_panel["crush_utilization"]) / 3.0)
soy_disagree = np.sign(soy_wf) * np.sign(soy_cargill) < 0.0
soy_final_signal = clean_signal(soy_wf.where(~soy_disagree, 0.50 * soy_wf))

soy_vol = soy_pnl["SOYABEAN"].rolling(60, min_periods=20).std().shift(1)
soy_low_vol = soy_vol < (0.80 * soy_vol.expanding(min_periods=252).median().shift(1))
soy_final_signal = clean_signal(soy_final_signal.where(~soy_low_vol.reindex(index).fillna(False), 0.0))

soy_outright_pos = positions_from_single_signal(soy_final_signal, soy_pnl, "SOYABEAN")
soy_calendar_pos = positions_from_single_signal(soy_final_signal, calendar_pnl[["SOYABEAN"]], "SOYABEAN")

display(soy_wf_rows)
display(pd.DataFrame({"final_rule": ["wf_momentum_mr + cargill_disagree_half + low_vol_flat"]}))

,year,selected
0,2018,trend_mom_else_mr
1,2019,mom60
2,2020,mom20


,final_rule
0,wf_momentum_mr + cargill_disagree_half + low_v...


## 5. Final Wheat Signal

In [6]:
pair_signals, _ = build_pair_signals(feature_panels, futures_pnl)
pair_price_trend = build_pair_price_trend_from_prices(data, futures_pnl)
mr_cargill_90_10 = 0.90 * pair_signals["pair_price_mr"] + 0.10 * pair_signals["pair_physical_cargill"]
trend_strength = pair_price_trend.abs()
trend_state = (trend_strength > trend_strength.expanding(min_periods=252).median().shift(1)).fillna(False)
conflict = trend_state & ((mr_cargill_90_10 * pair_price_trend) < 0.0)
wheat_final_signal = mr_cargill_90_10.where(~conflict, 0.0).fillna(0.0)

wheat_position_kwargs = {
    "target_daily_pair_vol": 40.0,
    "max_leg_lot": 0.40,
    "signal_threshold": 0.12,
    "halflife": 5.0,
    "rebalance_every": 5,
}
wheat_outright_pos = positions_from_pair_signal(wheat_final_signal, futures_pnl[["WHEAT_SRW", "WHEAT_HRW"]], **wheat_position_kwargs)
wheat_calendar_pos = positions_from_pair_signal(wheat_final_signal, calendar_pnl[["WHEAT_SRW", "WHEAT_HRW"]], **wheat_position_kwargs)

display(pd.DataFrame({"final_rule": ["SRW/HRW price MR + 10% Cargill physical, flat on strong trend conflict"]}))

,final_rule
0,"SRW/HRW price MR + 10% Cargill physical, flat ..."


## 6. Outright Futures-Only Combined Book

In [7]:
outright_columns = ["CORN", "SOYABEAN", "WHEAT_SRW", "WHEAT_HRW"]
outright_pnl = futures_pnl[outright_columns]
outright_sleeves = {
    "corn_outright": corn_outright_pos,
    "soybean_outright": soy_outright_pos,
    "wheat_pair_outright": wheat_outright_pos,
}

outright_rows = []
for name, pos in outright_sleeves.items():
    bt = backtest_book(pos.reindex(index).fillna(0.0), outright_pnl[pos.columns])
    outright_rows.append(result_row(name, bt))

combined_outright_pos = combine_equal_weight(outright_sleeves.values(), outright_columns)
combined_outright_bt = backtest_book(combined_outright_pos, outright_pnl)
outright_rows.append(result_row("combined_equal_weight_outright", combined_outright_bt))

outright_results = pd.DataFrame(outright_rows).sort_values("oos_sharpe", ascending=False)
display(outright_results.round(3))

,test,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,hit_rate,turnover,avg_gross_exposure
2,wheat_pair_outright,2.145,26.671,-0.170,75,1.462,51.553,-0.202,0.507,0.005,0.028
1,soybean_outright,1.893,668.552,-3.503,206,1.893,668.552,-3.503,0.558,0.005,0.044
3,combined_equal_weight_outright,1.661,302.048,-1.168,329,1.432,310.342,-1.168,0.550,0.004,0.016
0,corn_outright,1.413,215.627,-1.386,106,1.413,215.627,-1.386,0.481,0.028,0.059


## 7. Calendar-Spread Combined Book

In [8]:
calendar_columns = ["CORN", "SOYABEAN", "WHEAT_SRW", "WHEAT_HRW"]
calendar_book_pnl = calendar_pnl[calendar_columns]
calendar_sleeves = {
    "corn_calendar": corn_calendar_pos,
    "soybean_calendar": soy_calendar_pos,
    "wheat_pair_calendar": wheat_calendar_pos,
}

calendar_rows = []
for name, pos in calendar_sleeves.items():
    bt = backtest_book(pos.reindex(index).fillna(0.0), calendar_book_pnl[pos.columns])
    calendar_rows.append(result_row(name, bt))

combined_calendar_pos = combine_equal_weight(calendar_sleeves.values(), calendar_columns)
combined_calendar_bt = backtest_book(combined_calendar_pos, calendar_book_pnl)
calendar_rows.append(result_row("combined_equal_weight_calendar", combined_calendar_bt))

calendar_results = pd.DataFrame(calendar_rows).sort_values("oos_sharpe", ascending=False)
display(calendar_results.round(3))

,test,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,hit_rate,turnover,avg_gross_exposure
1,soybean_calendar,-0.347,-58.397,-3.916,206,-0.347,-58.397,-3.916,0.413,0.037,0.399
2,wheat_pair_calendar,-0.488,-17.686,-0.861,75,-1.022,-95.332,-1.492,0.440,0.059,0.341
3,combined_equal_weight_calendar,-0.744,-63.452,-1.666,334,-0.835,-89.334,-1.666,0.398,0.023,0.140
0,corn_calendar,-1.190,-88.953,-1.223,112,-1.190,-88.953,-1.223,0.268,0.087,0.363


## 8. Cross-Product Long/Short Book

This version converts the final outright product signals into a cross-sectional book. It is market-neutral by construction: long the strongest product signal and short the weakest product signal each day.

Two versions are shown:
- **ranked:** always ranks available products when there is signal dispersion;
- **disagreement-only:** trades only when the final product signals include both positive and negative views.

In [9]:
cross_columns = ["CORN", "SOYABEAN", "WHEAT_SRW", "WHEAT_HRW"]
cross_pnl = futures_pnl[cross_columns]

cross_scores = pd.DataFrame(0.0, index=index, columns=cross_columns)
cross_scores["CORN"] = corn_final_signal.reindex(index).fillna(0.0)
cross_scores["SOYABEAN"] = soy_final_signal.reindex(index).fillna(0.0)
cross_scores["WHEAT_SRW"] = wheat_final_signal.reindex(index).fillna(0.0)
cross_scores["WHEAT_HRW"] = -wheat_final_signal.reindex(index).fillna(0.0)

# Apply the same final guards before ranking, so the relative-value book uses the promoted outright signals.
cross_scores.loc[corn_guard.reindex(index).fillna(False), "CORN"] = 0.0
cross_scores.loc[soy_low_vol.reindex(index).fillna(False), "SOYABEAN"] = 0.0


def cross_product_positions(scores, disagreement_only=False, target_daily_vol=75.0, max_abs_lot=0.50):
    raw = scores.replace([np.inf, -np.inf], np.nan).fillna(0.0).copy()
    raw[raw.abs() < 0.05] = 0.0
    risk = cross_pnl.rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    risk_adjusted = raw / risk
    demeaned = risk_adjusted.sub(risk_adjusted.mean(axis=1), axis=0).fillna(0.0)

    if disagreement_only:
        has_long = (raw > 0.05).any(axis=1)
        has_short = (raw < -0.05).any(axis=1)
        demeaned = demeaned.where((has_long & has_short), 0.0)

    selected = pd.DataFrame(0.0, index=raw.index, columns=raw.columns)
    for date, row in demeaned.iterrows():
        row = row.dropna()
        if row.abs().max() <= 0.0:
            continue
        long_leg = row.idxmax()
        short_leg = row.idxmin()
        if long_leg == short_leg or row.loc[long_leg] <= 0.0 or row.loc[short_leg] >= 0.0:
            continue
        selected.loc[date, long_leg] = 1.0
        selected.loc[date, short_leg] = -1.0

    # Same spirit as the outright sleeve sizing: inverse PnL-vol lots, clipped, then product-sleeve scaled.
    lots = selected * (float(target_daily_vol) / risk)
    return (lots.clip(-float(max_abs_lot), float(max_abs_lot)) * PRODUCT_WEIGHT).fillna(0.0)


cross_ranked_pos = cross_product_positions(cross_scores, disagreement_only=False)
cross_disagreement_pos = cross_product_positions(cross_scores, disagreement_only=True)

cross_ranked_bt = backtest_book(cross_ranked_pos, cross_pnl)
cross_disagreement_bt = backtest_book(cross_disagreement_pos, cross_pnl)

cross_results = pd.DataFrame([
    result_row("cross_product_ranked_long_short", cross_ranked_bt),
    result_row("cross_product_disagreement_only", cross_disagreement_bt),
]).sort_values("oos_sharpe", ascending=False)

display(cross_results.round(3))

leg_counts = pd.DataFrame({
    "ranked_nonzero_days": (cross_ranked_pos.abs().sum(axis=1) > 0.0).resample("YE").sum(),
    "disagreement_nonzero_days": (cross_disagreement_pos.abs().sum(axis=1) > 0.0).resample("YE").sum(),
})
display(leg_counts.tail(5).round(0))

,test,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,hit_rate,turnover,avg_gross_exposure
0,cross_product_ranked_long_short,0.448,262.624,-4.571,553,-0.316,-407.530,-9.321,0.495,0.050,0.097
1,cross_product_disagreement_only,-0.479,-214.876,-5.090,451,-0.784,-885.031,-11.962,0.481,0.046,0.092


,ranked_nonzero_days,disagreement_nonzero_days
2016-12-31,158,158
2017-12-31,163,163
2018-12-31,199,149
2019-12-31,173,155
2020-12-31,181,146


## 8. Final Comparison

In [10]:
final_comparison = pd.DataFrame([
    result_row("combined_equal_weight_outright", combined_outright_bt),
    result_row("cross_product_ranked_long_short", cross_ranked_bt),
    result_row("cross_product_disagreement_only", cross_disagreement_bt),
    result_row("combined_equal_weight_calendar", combined_calendar_bt),
])
display(final_comparison.round(3))

o = final_comparison.loc[final_comparison["test"] == "combined_equal_weight_outright"].iloc[0]
r = final_comparison.loc[final_comparison["test"] == "cross_product_ranked_long_short"].iloc[0]
d = final_comparison.loc[final_comparison["test"] == "cross_product_disagreement_only"].iloc[0]
c = final_comparison.loc[final_comparison["test"] == "combined_equal_weight_calendar"].iloc[0]
display(Markdown(f"""
**Conclusion**

The combined book is strongest when the final product signals trade outright futures as independent sleeves.

- Outright futures-only: OOS Sharpe `{o["oos_sharpe"]:.3f}`, OOS PnL `{o["oos_pnl"]:.3f}`, OOS max DD `{o["oos_dd_pct"]:.3f}%`.
- Cross-product ranked long/short: OOS Sharpe `{r["oos_sharpe"]:.3f}`, OOS PnL `{r["oos_pnl"]:.3f}`, OOS max DD `{r["oos_dd_pct"]:.3f}%`.
- Cross-product disagreement only: OOS Sharpe `{d["oos_sharpe"]:.3f}`, OOS PnL `{d["oos_pnl"]:.3f}`, OOS max DD `{d["oos_dd_pct"]:.3f}%`.
- Calendar spread: OOS Sharpe `{c["oos_sharpe"]:.3f}`, OOS PnL `{c["oos_pnl"]:.3f}`, OOS max DD `{c["oos_dd_pct"]:.3f}%`.

Read: the product signals contain useful outright edge. Forcing them into cross-product relative value reduces performance, but it is still a cleaner diagnostic than calendar spreads because each leg is a different product outright future.
"""))

,test,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct,hit_rate,turnover,avg_gross_exposure
0,combined_equal_weight_outright,1.661,302.048,-1.168,329,1.432,310.342,-1.168,0.550,0.004,0.016
1,cross_product_ranked_long_short,0.448,262.624,-4.571,553,-0.316,-407.530,-9.321,0.495,0.050,0.097
2,cross_product_disagreement_only,-0.479,-214.876,-5.090,451,-0.784,-885.031,-11.962,0.481,0.046,0.092
3,combined_equal_weight_calendar,-0.744,-63.452,-1.666,334,-0.835,-89.334,-1.666,0.398,0.023,0.140



**Conclusion**

The combined book is strongest when the final product signals trade outright futures as independent sleeves.

- Outright futures-only: OOS Sharpe `1.661`, OOS PnL `302.048`, OOS max DD `-1.168%`.
- Cross-product ranked long/short: OOS Sharpe `0.448`, OOS PnL `262.624`, OOS max DD `-4.571%`.
- Cross-product disagreement only: OOS Sharpe `-0.479`, OOS PnL `-214.876`, OOS max DD `-5.090%`.
- Calendar spread: OOS Sharpe `-0.744`, OOS PnL `-63.452`, OOS max DD `-1.666%`.

Read: the product signals contain useful outright edge. Forcing them into cross-product relative value reduces performance, but it is still a cleaner diagnostic than calendar spreads because each leg is a different product outright future.
